# 第一課 - AI 代理簡介

歡迎來到<strong>AI 代理入門</strong>課程的第一課！

<strong>AI 代理</strong>是一個使用大型語言模型（LLM）作為推理引擎的程式，能在現實世界中執行<em>行動</em>— 呼叫 API、查詢資料庫或執行程式碼 — 以代表使用者完成目標。

在本筆記本中，你將建立你的第一個代理：一個推薦度假目的地的<strong>旅遊代理</strong>。過程中你將學習如何：

1. 使用<strong>Microsoft Agent Framework</strong>連接到 Microsoft Foundry Agent 服務。
2. 為代理提供一個<strong>工具</strong> — 它可以呼叫的純 Python 函數。
3. 執行代理並檢視其回應。
4. 一個接一個地串流代理的回應標記。


## 設定

在執行此筆記本之前，請確保您已經：

1. **擁有一個已部署聊天模型的 Microsoft Foundry 專案**（例如 `gpt-4.1-mini`）。
2. **已使用 Azure CLI 登入** — 在您的終端機執行 `az login`。
3. **設定所需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Microsoft Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名稱。

下面的程式碼格將安裝您所需的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 創建你的第一個代理人

一個代理人需要兩樣東西：

- <strong>指令</strong> 告訴它 <em>它是誰</em> 以及 <em>如何行為</em>（系統提示）。
- <strong>工具</strong> — 使用 `@tool` 裝飾的 Python 函數，代理人可以調用這些函數來獲取資訊或執行操作。

以下我們定義了一個簡單的工具，返回受歡迎的度假勝地列表。當用戶要求旅行建議時，代理人將會使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 串流回應

為了提供更具互動性的體驗，你可以 <strong>串流</strong> 代理的回應。代理會隨著生成而輸出文本片段，而不是等待完整的回覆。這在想要即時顯示輸出的聊天介面中尤其有用。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 摘要

在本課程中，您學會了如何：

- <strong>建立一個提供者</strong>，透過 `FoundryChatClient` 連接到 Microsoft Foundry Agent Service。
- **使用 `@tool` 裝飾器定義工具**，讓代理可以呼叫您的 Python 函式。
- <strong>執行代理</strong> 並輸入使用者訊息，然後印出其回應。
- <strong>串流回應</strong> 以進行即時輸出。

在下一課中，我們將更深入探索代理框架，並學習如何賦予代理更強大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
